# Was 1900 a leap year? Excel says yes

A three-condition rule, and a spreadsheet that gets it wrong on purpose.

In [1]:
def is_leap_naive(year):
    return year % 4 == 0

def is_leap(year):
    return year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)

for y in [1900, 2000, 2024, 2100]:
    print(y, "naive:", is_leap_naive(y), "  correct:", is_leap(y))

1900 naive: True   correct: False
2000 naive: True   correct: True
2024 naive: True   correct: True
2100 naive: True   correct: False


## Write the parentheses

`A and B or C` is read by Python as `(A and B) or C`, because `and` binds
tighter than `or`. Here the two groupings happen to agree, because a year that
is divisible by 400 is also divisible by 4; the cell below finds no year on
which they differ. That is luck rather than design, and a reader who assumes
the other grouping reads a different rule.

In [2]:
def is_leap_no_parentheses(year):
    return year % 4 == 0 and year % 100 != 0 or year % 400 == 0   # same tokens, no parentheses

print([y for y in range(1896, 2104) if is_leap(y) != is_leap_no_parentheses(y)])

[]


## Check against the standard library

In [3]:
import calendar
print(all(is_leap(y) == calendar.isleap(y) for y in range(1, 3000)))

True


## The Excel bug

Excel treats 1900 as a leap year for compatibility with Lotus 1-2-3. Its serial
number 60 is the non-existent 29 February 1900, so the day count before
1 March 1900 is off by one.

In [4]:
from datetime import date, timedelta

def excel_serial_to_date(serial):
    """Excel's 1900 date system, bug included."""
    if serial < 60:
        return date(1899, 12, 31) + timedelta(days=serial)
    if serial == 60:
        return "1900-02-29 (does not exist)"
    return date(1899, 12, 30) + timedelta(days=serial)   # from March 1900 on, the offset absorbs the bug

for s in [1, 59, 60, 61, 45000]:
    print(s, excel_serial_to_date(s))

1 1900-01-01
59 1900-02-28
60 1900-02-29 (does not exist)
61 1900-03-01
45000 2023-03-15


**What we used:** conditionals, operator precedence and control flow
(session 4, sections 4 and 8); flowcharts (session 5, section 5).